In [ ]:
import os

#----Dev Note: the three lines below are device specific. Without it, Tensorflow crashes and burns.
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["TF_XLA_FLAGS"] = "--tf_xla_enable_xla_devices=false"


import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import keras
import cv2
import pandas as pd
with_mask_path = os.path.join("dataset","data/with_mask")
without_mask_path = os.path.join("dataset","data/without_mask")

data = []

for img_file in os.listdir(with_mask_path):
    img_path = os.path.join(with_mask_path, img_file)
    image = cv2.imread(img_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (224,224))
    image = np.array(image)
    data.append({
        "image": image,
        "label": 1,
    })


for img_file in os.listdir(without_mask_path):
    img_path = os.path.join(without_mask_path, img_file)
    image = cv2.imread(img_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (224,224))
    image = np.array(image)
    data.append({
        "image": image,
        "label": 0,
    })


df = pd.DataFrame(data)


X = np.array(df["image"].tolist())
Y = np.array(df["label"].tolist())

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, stratify=Y,random_state=42)

X_train_scaled = X_train.astype("float32") / 255.0
x_test_scaled = X_test.astype("float32") / 255.0

aug_data =  keras.models.Sequential([
            
            keras.layers.RandomRotation(factor=0.055),
            keras.layers.RandomZoom(height_factor=0.15),
            keras.layers.RandomTranslation(height_factor=0.2, width_factor=0.2),
            keras.layers.RandomFlip("horizontal"),

            ])

base_model = keras.applications.MobileNetV2(

    weights="imagenet",

    include_top=False,

    input_shape=(224, 224, 3)

)

base_model.trainable = False


model = keras.Sequential(
    [
        aug_data,
        base_model,
        keras.layers.Flatten(),

        keras.layers.Dense(256, activation="relu"),

        keras.layers.Dropout(0.5),
        keras.layers.Dense(2, activation="softmax")
    ]
)

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["acc"],jit_compile=False)
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss', 
    patience=5,          
    restore_best_weights=True
)


early_stop = keras.callbacks.EarlyStopping(

    monitor="val_loss",

    patience=3,

    restore_best_weights=True

)

print("\nStarting Phase 1 Training...")


history = model.fit(

    X_train_scaled,

    y_train,

    validation_split=0.1,

    epochs=5,

    batch_size=16,

    callbacks=[early_stop]

)

print("\nStarting Fine-Tuning...")

base_model.trainable = True

# Freeze earlier layers
for layer in base_model.layers[:-20]:
    layer.trainable = False

# Recompile with lower learning rate
model.compile(

    optimizer=keras.optimizers.Adam(1e-5),

    loss="binary_crossentropy",

    metrics=["accuracy"]

)

fine_tune_history = model.fit(

    X_train_scaled,

    y_train,

    validation_split=0.1,

    epochs=5,

    batch_size=16,

    callbacks=[early_stop]

)

loss, accuracy = model.evaluate(x_test_scaled, y_test)

print(f"\nTest Accuracy: {accuracy:.4f}")

model.save("models/mask_detector_transfer_learning.keras")

print("\nModel Saved Successfully!")

plt.subplot(1,2,1)

plt.plot(history.history["loss"], label="Train Loss")

plt.plot(history.history["val_loss"], label="Validation Loss")

plt.title("Training vs Validation Loss")

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.legend()

plt.subplot(1,2,2)

plt.plot(history.history["accuracy"], label="Train Accuracy")

plt.plot(history.history["val_accuracy"], label="Validation Accuracy")

plt.title("Training vs Validation Accuracy")

plt.xlabel("Epoch")

plt.ylabel("Accuracy")

plt.legend()

plt.tight_layout()

plt.show()

